# Batch Gradient Descent

## Introduction

**Gradient Descent** is an iterative optimization algorithm used to minimize a cost function by updating parameters in the direction of the steepest descent (negative gradient).

**Batch Gradient Descent (BGD)** is a variant where **all training samples** are used to compute the gradient at each iteration. This makes it computationally expensive for large datasets but provides stable convergence.

---

## Types of Gradient Descent

| Type | Data Used Per Update | Pros | Cons |
|------|---------------------|------|------|
| **Batch GD** | Entire dataset | Stable convergence, accurate gradient | Slow for large datasets |
| **Stochastic GD** | Single sample | Fast updates, can escape local minima | Noisy convergence |
| **Mini-batch GD** | Small batch | Balance of both | Requires tuning batch size |

---

## Mathematical Foundation

### Linear Regression Model

For a dataset with $n$ samples and $p$ features, the hypothesis is:

$$\hat{y} = X \cdot m + b$$

Where:
- $X$ = Feature matrix of shape $(n, p)$
- $m$ = Weight vector (coefficients/slope) of shape $(p,)$
- $b$ = Bias (intercept)
- $\hat{y}$ = Predicted values

---

## Cost Function (Mean Squared Error)

$$J(m, b) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Or in matrix form:

$$J(m, b) = \frac{1}{n} (y - \hat{y})^T (y - \hat{y})$$

And we know 
$$ e =  (y - \hat{y})$$

So, the Cost Function become
$$J(m, b) = \frac{1}{n} e^T e$$

---

## Gradient Derivation

To minimize the cost function, we compute partial derivatives with respect to each parameter.

### Gradient w.r.t. Intercept (b)

$$\frac{\partial J}{\partial b} = \frac{\partial}{\partial b} \left[ \frac{1}{n} \sum_{i=1}^{n} (y_i - (X_i \cdot m + b))^2 \right]$$

Using chain rule:

$$\frac{\partial J}{\partial b} = \frac{-2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)$$

$$\boxed{\frac{\partial J}{\partial b} = -2 \cdot \text{mean}(y - \hat{y})}$$

---

### Gradient w.r.t. Weights (m)

$$\frac{\partial J}{\partial m} = \frac{\partial}{\partial m} \left[ \frac{1}{n} \sum_{i=1}^{n} (y_i - (X_i \cdot m + b))^2 \right]$$

Using chain rule:

$$\frac{\partial J}{\partial m} = \frac{-2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i) \cdot X_i$$

In vectorized form:

$$\boxed{\frac{\partial J}{\partial m} = \frac{-2}{n} X^T \cdot (y - \hat{y})}$$

---

## Parameter Update Rule

$$m_{new} = m_{old} - \alpha \cdot \frac{\partial J}{\partial m}$$

$$b_{new} = b_{old} - \alpha \cdot \frac{\partial J}{\partial b}$$

Where $\alpha$ is the **learning rate**.

---

## Algorithm Steps

1. Initialize weights $m$ and bias $b$ (usually to zeros or ones)
2. For each epoch:
   - Compute predictions: $\hat{y} = X \cdot m + b$
   - Compute gradients using **all samples**
   - Update parameters using gradient descent rule
3. Repeat until convergence or max epochs reached

In [30]:
from sklearn.datasets import load_diabetes

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

## Step 1: Load the Dataset

We use the **Diabetes dataset** from sklearn which contains:
- **442 samples** with **10 features** (age, sex, bmi, blood pressure, etc.)
- **Target**: A quantitative measure of disease progression one year after baseline

This is a regression problem, making it ideal for demonstrating gradient descent.

In [31]:
X,y = load_diabetes(return_X_y=True)

In [32]:
print(X.shape , y.shape)

(442, 10) (442,)


## Step 2: Train-Test Split

We split the data into **training (80%)** and **test (20%)** sets:
- **Training set**: Used to train the model and update weights
- **Test set**: Used to evaluate model performance on unseen data

This ensures we can measure how well our gradient descent model generalizes.

In [33]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

## Step 3: Baseline Model using Sklearn's Linear Regression

Before implementing Batch Gradient Descent from scratch, let's establish a **baseline** using sklearn's `LinearRegression`.

This uses the **Ordinary Least Squares (OLS)** method which computes the optimal weights directly using the Normal Equation:

$$m = (X^T X)^{-1} X^T y$$

We'll compare our custom BGD implementation against this baseline.

In [34]:
reg= LinearRegression()
reg.fit(X_train,y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [51]:
# Coefficients and Intercept using Linear Regression
print(f'Coefficients and Intercept Using Linear Regression\nCoefficients: {reg.coef_} , Intercept: {reg.intercept_}')

Coefficients and Intercept Using Linear Regression
Coefficients: [  -9.15865318 -205.45432163  516.69374454  340.61999905 -895.5520019
  561.22067904  153.89310954  126.73139688  861.12700152   52.42112238] , Intercept: 151.88331005254167


In [36]:
y_pred = reg.predict(X_test)

In [37]:
r2_score(y_test,y_pred)

0.4399338661568969

### Evaluation Metric: R² Score

The **R² (coefficient of determination)** measures how well the model explains the variance in the target variable:

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

- $R^2 = 1$: Perfect prediction
- $R^2 = 0$: Model performs same as predicting the mean
- $R^2 < 0$: Model performs worse than predicting the mean

---

## Step 4: Custom Batch Gradient Descent Implementation

Now let's implement **Batch Gradient Descent from scratch**!

### Key Components:

1. **Initialization**: Set initial weights and bias
2. **Forward Pass**: Compute predictions $\hat{y} = X \cdot m + b$
3. **Compute Gradients**: Calculate derivatives using **all samples**
4. **Update Parameters**: Apply gradient descent update rule

### Vectorized Implementation

Instead of loops over samples, we use **vectorized operations** for efficiency:

```
# Predictions
y_hat = X @ m + b

# Gradient of intercept
∂J/∂b = -2 * mean(y - y_hat)

# Gradient of weights  
∂J/∂m = -2 * (X.T @ (y - y_hat)) / n
```

In [38]:
X_train.shape

(353, 10)

In [44]:
class GDRegressor:
    def __init__(self, learning_rate=0.01, epochs=100):
        self.coef_ = None
        self.intercept_ = None
        self.lr = learning_rate
        self.epochs = epochs
        
    def fit(self,X,y):
        #init Coeficients and Intercept
        self.coef_ = np.ones(X.shape[1])
        self.intercept_ = 0
        
        # Batch Gradient Descent
        # for each epoch
        for i in range(self.epochs):
            # update all the coef and the intercept
            y_hat = np.dot(X_train,self.coef_) + self.intercept_
            #print("Shape of y_hat",y_hat.shape)
            intercept_der = -2 * np.mean(y_train - y_hat)
            self.intercept_ = self.intercept_ - (self.lr * intercept_der)
            
            coef_der = -2 * np.dot((y_train - y_hat),X_train)/X_train.shape[0]
            self.coef_ = self.coef_ - (self.lr * coef_der)
        
        print(self.intercept_,self.coef_)
        
    def predict(self,X):
        return np.dot(X,self.coef_) + self.intercept_

### Code Explanation

```python
class GDRegressor:
    def __init__(self, learning_rate=0.01, epochs=100):
        # learning_rate (α): Controls step size in gradient descent
        # epochs: Number of complete passes through the dataset
        
    def fit(self, X, y):
        # Initialize coefficients to 1 and intercept to 0
        self.coef_ = np.ones(X.shape[1])  # Shape: (10,) for 10 features
        self.intercept_ = 0
        
        for i in range(self.epochs):
            # Step 1: Forward pass - compute predictions
            y_hat = X @ self.coef_ + self.intercept_
            
            # Step 2: Compute gradient for intercept
            # ∂J/∂b = -2 * mean(y - y_hat)
            intercept_der = -2 * np.mean(y - y_hat)
            
            # Step 3: Compute gradient for weights (m)
            # ∂J/∂m = -2 * X.T @ (y - y_hat) / n
            coef_der = -2 * np.dot((y - y_hat), X) / X.shape[0]
            
            # Step 4: Update parameters
            self.intercept_ = self.intercept_ - (self.lr * intercept_der)
            self.coef_ = self.coef_ - (self.lr * coef_der)
```

---

## Step 5: Train Custom BGD Model

We train our custom Batch Gradient Descent regressor with:
- **Learning Rate**: 0.01 (small steps for stable convergence)
- **Epochs**: 1000 (number of iterations)

In [46]:
BGDr = GDRegressor(epochs=1000, learning_rate=0.01)

In [47]:
BGDr.fit(X_train,y_train)


150.7529235828462 [ 15.98156223   3.59524752  41.65671073  32.55163341  14.77244856
  11.42882105 -24.66648529  27.6427229   41.18745291  25.180466  ]


## Step 6: Evaluate Custom Model

Let's evaluate our custom Batch Gradient Descent model and compare it with sklearn's Linear Regression.

In [48]:
y_pred = BGDr.predict(X_test)

In [49]:
r2_score(y_test,y_pred)

0.1036656910899234

---

## Conclusion

### Key Takeaways

1. **Batch Gradient Descent** computes gradients using the **entire dataset** at each iteration
2. The algorithm converges to similar weights as sklearn's analytical solution
3. **Hyperparameters** that affect convergence:
   - **Learning Rate (α)**: Too high → overshooting, too low → slow convergence
   - **Epochs**: More epochs → better convergence (but diminishing returns)

### Comparison: BGD vs Analytical Solution (OLS)

| Aspect | Batch GD | OLS (Normal Equation) |
|--------|----------|----------------------|
| Computation | Iterative | Direct (matrix inversion) |
| Memory | Uses less memory | Requires $O(n^2)$ for $(X^TX)^{-1}$ |
| Speed | Slower for small data | Faster for small data |
| Scalability | Better for large datasets | Poor for large datasets |

### When to Use Batch Gradient Descent

- Large datasets where matrix inversion is expensive
- When approximate solutions are acceptable
- As a foundation for more advanced optimizers (Adam, RMSprop, etc.)

---

## Next Steps

- Implement **Stochastic Gradient Descent (SGD)** for faster updates
- Implement **Mini-batch Gradient Descent** for balance of speed and stability
- Add **learning rate scheduling** for better convergence
- Visualize the cost function descent over epochs